# Two-Layer Neural Network: Forward and Backward Propagation

This notebook demonstrates the implementation of a simple two-layer neural network from scratch using NumPy. It covers forward propagation, backward propagation, and training on a toy dataset. Explanations and improved variable names are provided throughout for clarity.

### Forward propagation:

## Forward Propagation Explained

Forward propagation computes the output of the neural network given the input data. It involves applying linear transformations and activation functions layer by layer, resulting in class probabilities at the output.

```python

# Linear transformation for the first (hidden) layer
hidden_linear = input_data @ weights_input_hidden + bias_hidden
# Apply ReLU activation to introduce non-linearity
hidden_activation = np.maximum(0, hidden_linear)
# Linear transformation for the output layer
output_linear = hidden_activation @ weights_hidden_output + bias_output
# Compute softmax probabilities for classification
exp_scores = np.exp(output_linear)
probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
```

```python

# Compute gradient of loss w.r.t. output layer scores
output_error = probabilities
output_error[np.arange(len(input_data)), true_labels] -= 1
# Gradients for output layer weights and biases
grad_weights_hidden_output = hidden_activation.T @ output_error
grad_bias_output = np.sum(output_error, axis=0)
# Backpropagate error to hidden layer
hidden_error = (output_error @ weights_hidden_output.T) * (hidden_activation > 0)
# Gradients for input-to-hidden weights and biases
grad_weights_input_hidden = input_data.T @ hidden_error
grad_bias_hidden = np.sum(hidden_error, axis=0)
```

## Backward Propagation Explained

Backward propagation computes the gradients of the loss with respect to each parameter in the network. These gradients are used to update the weights and biases to minimize the loss during training.

**Variable explanations:**

- `input_data`: Input matrix, shape (num_samples, input_dim)
- `weights_input_hidden`: Weights from input to hidden layer, shape (input_dim, hidden_dim)
- `bias_hidden`: Bias for hidden layer, shape (hidden_dim,)
- `hidden_activation`: Output of hidden layer after ReLU, shape (num_samples, hidden_dim)
- `weights_hidden_output`: Weights from hidden to output layer, shape (hidden_dim, output_dim)
- `bias_output`: Bias for output layer, shape (output_dim,)
- `output_linear`: Weighted sum at output layer, shape (num_samples, output_dim)
- `exp_scores`: Exponentiated output scores for softmax
- `probabilities`: Output probabilities for each class
- `true_labels`: True class labels, shape (num_samples,)

These variables are used throughout the forward and backward propagation steps.

## Code

## Implementation: Two-Layer Neural Network

Below is the implementation of a two-layer neural network using improved variable names and detailed comments. This class supports forward propagation, loss calculation, and training with gradient descent.

In [27]:
import numpy as np

class TwoLayerNeuralNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        # Initialize weights and biases
        self.weights_input_hidden = np.random.randn(input_dim, hidden_dim) * 0.01
        self.bias_hidden = np.zeros(hidden_dim)
        self.weights_hidden_output = np.random.randn(hidden_dim, output_dim) * 0.01
        self.bias_output = np.zeros(output_dim)

    def forward(self, input_data):
        # First layer: Linear + ReLU
        self.hidden_linear = input_data @ self.weights_input_hidden + self.bias_hidden
        self.hidden_activation = np.maximum(0, self.hidden_linear)
        # Output layer: Linear + Softmax
        self.output_linear = self.hidden_activation @ self.weights_hidden_output + self.bias_output
        exp_scores = np.exp(self.output_linear - np.max(self.output_linear, axis=1, keepdims=True))
        self.probabilities = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
        return self.probabilities

    def compute_loss(self, input_data, true_labels):
        probs = self.forward(input_data)
        num_samples = input_data.shape[0]
        correct_logprobs = -np.log(probs[np.arange(num_samples), true_labels])
        data_loss = np.sum(correct_logprobs) / num_samples
        return data_loss

    def train(self, input_data, true_labels, num_epochs, learning_rate=0.1):
        num_samples = input_data.shape[0]
        for epoch in range(num_epochs):
            # Forward pass
            probs = self.forward(input_data)
            # Backward pass
            output_error = probs
            output_error[np.arange(num_samples), true_labels] -= 1
            output_error /= num_samples
            grad_weights_hidden_output = self.hidden_activation.T @ output_error
            grad_bias_output = np.sum(output_error, axis=0)
            hidden_error = (output_error @ self.weights_hidden_output.T) * (self.hidden_activation > 0)
            grad_weights_input_hidden = input_data.T @ hidden_error
            grad_bias_hidden = np.sum(hidden_error, axis=0)
            # Update parameters
            self.weights_input_hidden -= learning_rate * grad_weights_input_hidden
            self.bias_hidden -= learning_rate * grad_bias_hidden
            self.weights_hidden_output -= learning_rate * grad_weights_hidden_output
            self.bias_output -= learning_rate * grad_bias_output
            # Print loss every 100 epochs
            if epoch % 100 == 0:
                loss = self.compute_loss(input_data, true_labels)
                print(f"Epoch {epoch}: loss = {loss:.4f}")

This class implements a two-layer neural network with clear variable names and comments. The `forward` method computes the output probabilities, `compute_loss` calculates the cross-entropy loss, and `train` performs parameter updates using gradient descent.

### Test

## Example: Training and Testing on a Toy Dataset

Let's see how to use the `TwoLayerNeuralNet` class to train and test the network on a simple dataset. The following cell generates a toy dataset, trains the network, and prints predictions.

Here's an example of how to use the TwoLayerNet class to train and test the network on a toy dataset:

In [28]:
# Generate a simple XOR-like dataset
input_data = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
true_labels = np.array([0, 1, 1, 0])

# Initialize the neural network
net = TwoLayerNeuralNet(input_dim=2, hidden_dim=10, output_dim=2)

# Train the neural network
net.train(input_data, true_labels, num_epochs=1000, learning_rate=0.1)

# Test the neural network
probs = net.forward(input_data)
predictions = np.argmax(probs, axis=1)
print("Predictions:", predictions)
print("True labels:", true_labels)

Epoch 0: loss = 0.6931
Epoch 100: loss = 0.6928
Epoch 200: loss = 0.6894
Epoch 300: loss = 0.6566
Epoch 400: loss = 0.5461
Epoch 500: loss = 0.4692
Epoch 600: loss = 0.3966
Epoch 700: loss = 0.2592
Epoch 800: loss = 0.1224
Epoch 900: loss = 0.0685
Predictions: [0 1 1 0]
True labels: [0 1 1 0]


### Improvements 

There are several ways to improve the implementation of a two-layer neural network with softmax. Here are a few suggestions:

1. Weight initialization: The current implementation initializes the weights randomly using a Gaussian distribution. However, it is recommended to use other weight initialization methods such as Xavier or He initialization to improve convergence and avoid vanishing or exploding gradients. One possible implementation for Xavier initialization of the weights is:

## Further Improvements

There are several ways to improve the basic two-layer neural network implementation. The following sections describe and demonstrate some of these enhancements, including better weight initialization, learning rate decay, regularization, mini-batch training, and advanced optimizers.

In [29]:
# Xavier initialization for better convergence
# self.weights_input_hidden = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
# self.weights_hidden_output = np.random.randn(hidden_dim, output_dim) / np.sqrt(hidden_dim)

2. Learning rate decay: The learning rate is a hyperparameter that determines the step size at each iteration during training. However, using a fixed learning rate may lead to suboptimal performance or slow convergence. A common technique is to gradually decrease the learning rate over time, known as learning rate decay, to fine-tune the network weights as the optimization process progresses.

In [30]:
# Learning rate decay example
# learning_rate = 0.1
# lr_decay = 0.95
# lr_decay_epoch = 100
# for epoch in range(num_epochs):
#     # ...training code...
#     if epoch % lr_decay_epoch == 0:
#         learning_rate *= lr_decay

3. Regularization: Overfitting can occur when the model is too complex and the training data is limited. Regularization techniques such as L1 or L2 regularization can be applied to the loss function to prevent overfitting and improve the generalization performance of the model.


In [31]:
# L2 regularization example
# reg_lambda = 0.1
# data_loss += 0.5 * reg_lambda * (np.sum(self.weights_input_hidden ** 2) + np.sum(self.weights_hidden_output ** 2))

4. Mini-batch training: The current implementation updates the weights using the entire training set at each iteration, which can be computationally expensive for large datasets. An alternative is to use mini-batch training, where a random subset of the training data is used at each iteration to update the weights. This can speed up the training process and improve convergence.

In [32]:
# Mini-batch training example
# batch_size = 64
# num_batches = len(input_data) // batch_size
# for epoch in range(num_epochs):
#     for i in range(num_batches):
#         # Select a random batch
#         batch_indices = np.random.choice(len(input_data), batch_size)
#         X_batch = input_data[batch_indices]
#         y_batch = true_labels[batch_indices]
#         # ...forward and backward pass on batch...

5. Optimization algorithm: The current implementation uses stochastic gradient descent (SGD) as the optimization algorithm. However, there are other optimization algorithms such as Adam, Adagrad, and RMSprop that can improve the convergence speed and performance of the network.

In [33]:
# # Adam optimizer update example
# beta1, beta2 = 0.9, 0.999
# eps = 1e-8
# m_wih, v_wih = 0, 0
# m_who, v_who = 0, 0
# for epoch in range(num_epochs):
#     # ...forward and backward pass...
#     m_wih = beta1 * m_wih + (1 - beta1) * grad_weights_input_hidden
#     v_wih = beta2 * v_wih + (1 - beta2) * (grad_weights_input_hidden ** 2)
#     m_who = beta1 * m_who + (1 - beta1) * grad_weights_hidden_output
#     v_who = beta2 * v_who + (1 - beta2) * (grad_weights_hidden_output ** 2)
#     self.weights_input_hidden -= learning_rate * m_wih / (np.sqrt(v_wih) + eps)
#     self.bias_hidden -= learning_rate * grad_bias_hidden
#     self.weights_hidden_output -= learning_rate * m_who / (np.sqrt(v_who) + eps)
#     self.bias_output -= learning_rate * grad_bias_output

## Other Extensions

You can further extend this implementation to support arbitrary activation functions, loss functions, and deeper (multi-layer) networks. See below for a flexible class structure.

In [34]:
import numpy as np

class ActivationFunction:
    def __call__(self, x):
        raise NotImplementedError
    def derivative(self, x):
        raise NotImplementedError

class ReLU(ActivationFunction):
    def __call__(self, x):
        return np.maximum(0, x)
    def derivative(self, x):
        return (x > 0).astype(float)

class Softmax(ActivationFunction):
    def __call__(self, x):
        exp_scores = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    def derivative(self, x):
        raise NotImplementedError

class MultiLayerNeuralNet:
    def __init__(self, input_dim, hidden_dims, output_dim, activation_fn, loss_fn, reg_lambda=0.0):
        self.num_layers = 1 + len(hidden_dims)
        self.layer_sizes = [input_dim] + hidden_dims + [output_dim]
        self.weights = {}
        self.biases = {}
        for i in range(1, self.num_layers + 1):
            self.weights[i] = np.random.randn(self.layer_sizes[i-1], self.layer_sizes[i]) / np.sqrt(self.layer_sizes[i-1])
            self.biases[i] = np.zeros(self.layer_sizes[i])
        self.activation_fn = activation_fn
        self.loss_fn = loss_fn
        self.reg_lambda = reg_lambda
        
    def forward(self, X):
        # Initialize dictionaries with layer 0 (input)
        self.layer_inputs = {}   # Pre-activation values (z)
        self.layer_outputs = {0: X}  # Post-activation values (activations)
        
        out = X
        for i in range(1, self.num_layers + 1):
            z = out @ self.weights[i] + self.biases[i]
            self.layer_inputs[i] = z      # Store pre-activation for layer i
            out = self.activation_fn(z)
            self.layer_outputs[i] = out   # Store post-activation for layer i
            
        return out
    
    def backward(self, X, y, output):
        delta = output - y
        grads_w = {}
        grads_b = {}
        delta /= X.shape[0]
        
        for i in reversed(range(1, self.num_layers + 1)):
            z = self.layer_inputs[i]  # Access by layer number
            activation_deriv = self.activation_fn.derivative(z)
            
            # Use layer_outputs[i-1] to get previous layer's output
            grads_w[i] = self.layer_outputs[i-1].T @ delta + self.reg_lambda * self.weights[i]
            grads_b[i] = np.sum(delta, axis=0)
            
            delta = (delta @ self.weights[i].T) * activation_deriv
            
        return grads_w, grads_b
    
    def loss(self, X, y, output):
        data_loss = self.loss_fn(output, y)
        reg_loss = sum(0.5 * self.reg_lambda * np.sum(self.weights[i] ** 2) for i in range(1, self.num_layers + 1))
        return data_loss + reg_loss
    
    def train(self, X, y, num_epochs, learning_rate=0.1):
        for epoch in range(num_epochs):
            output = self.forward(X)
            grads_w, grads_b = self.backward(X, y, output)
            
            for i in range(1, self.num_layers + 1):
                self.weights[i] -= learning_rate * grads_w[i]
                self.biases[i] -= learning_rate * grads_b[i]
                
            if epoch % 100 == 0:
                print(f"Epoch {epoch}, loss: {self.loss(X, y, output):.4f}")
    
    def debug_layers(self):
        """Helper method to inspect layer shapes"""
        print("\n=== Layer Information ===")
        print(f"Layer 0 (Input): shape = {self.layer_outputs[0].shape}")
        for i in range(1, self.num_layers + 1):
            print(f"\nLayer {i}:")
            print(f"  Pre-activation (z): shape = {self.layer_inputs[i].shape}")
            print(f"  Post-activation (a): shape = {self.layer_outputs[i].shape}")
            print(f"  Weights: shape = {self.weights[i].shape}")
            print(f"  Biases: shape = {self.biases[i].shape}")


### Test

## Example: Multi-Layer Neural Network on Synthetic Data

The following cell demonstrates how to use the flexible `MultiLayerNeuralNet` class on a synthetic dataset. It includes data normalization and accuracy evaluation.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Generate a synthetic classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=42)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize features
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Define mean squared error loss for demonstration
def mse_loss(output, y):
    return np.mean((output.flatten() - y) ** 2)

# Use sigmoid for binary output
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
class Sigmoid(ActivationFunction):
    def __call__(self, x):
        return sigmoid(x)
    def derivative(self, x):
        s = sigmoid(x)
        return s * (1 - s)
# Cross-entropy loss function
def cross_entropy_loss(output, y):
    m = y.shape[0]
    log_likelihood = -np.log(output[range(m), y.argmax(axis=1)])
    return np.sum(log_likelihood) / m
# Create and train a multi-layer neural network
nn = MultiLayerNeuralNet(
        input_dim=10,
        hidden_dims=[64, 32],
        output_dim=2,
        activation_fn=ReLU(),
        loss_fn=cross_entropy_loss,
        reg_lambda=0.01
    )
    
# Train for a few epochs
nn.train(X, y, num_epochs=500, learning_rate=0.01)

# Debug layer information
nn.debug_layers()

# Access specific layers during debugging
print("\n=== Debugging Specific Layers ===")
print(f"Layer 1 pre-activation mean: {nn.layer_inputs[1].mean():.4f}")
print(f"Layer 2 post-activation mean: {nn.layer_outputs[2].mean():.4f}")

ValueError: operands could not be broadcast together with shapes (1000,2) (1000,) 


* Arbitrary activation function 
* Arbitrary loss function 
* Extension to multiple layers